# 6장. 데이터를 보며 질문을 만드는 EDA

이 노트북은 `book/chapters/ch06_eda_questions.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 그래프나 표를 많이 만드는 것이 아니라, 데이터를 보며 **현재 데이터로 답할 수 있는 질문**을 만들고 그 질문을 지표와 코드로 연결하는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 노트북은 5장에서 만든 `data/processed/*_clean.csv` 파일을 사용합니다.
- 전처리 파일이 없다면 먼저 `python scripts/preprocess_data.py`를 실행하세요.
- EDA 결과표는 `reports/` 폴더에 CSV와 Markdown 보고서로 저장합니다.
- EDA 결과는 최종 결론이 아니라, 다음 분석 질문을 만드는 중간 산출물입니다.


## 1. EDA는 결론보다 질문에 가깝다

EDA는 Exploratory Data Analysis, 즉 탐색적 데이터 분석입니다. 이름 그대로 데이터를 여러 방향에서 탐색하면서 분포, 패턴, 차이, 관계, 이상한 값을 발견하는 과정입니다.

EDA에서 자주 던지는 질문은 다음과 같습니다.

- 데이터는 어떤 변수들로 구성되어 있는가?
- 주요 숫자형 변수의 분포는 어떤가?
- 주요 범주형 변수의 빈도는 어떤가?
- 특정 그룹 간 차이가 있는가?
- 시간에 따른 변화가 있는가?
- 이상하게 큰 값이나 작은 값이 있는가?
- 다음 단계에서 더 깊이 볼 질문은 무엇인가?


## 2. 관찰, 가설, 결론 구분하기

EDA 결과를 해석할 때는 관찰, 가설, 결론을 구분해야 합니다.

| 구분 | 의미 | 예시 |
|---|---|---|
| 관찰 | 데이터에서 직접 확인한 사실 | 전자기기 카테고리의 매출 비중이 가장 높다 |
| 가설 | 관찰을 바탕으로 생각해 볼 가능성 | 전자기기는 단가가 높아 매출 비중이 클 수 있다 |
| 결론 | 추가 검증 후 말할 수 있는 판단 | 단가와 판매 수량을 함께 분석한 결과 매출 차이의 주요 요인은 단가였다 |

이번 장에서는 주로 관찰과 가설을 만들고, 다음 분석 질문으로 연결하는 연습을 합니다.


## 3. 좋은 분석 질문의 조건

좋은 분석 질문은 데이터로 답할 수 있어야 합니다.

| 조건 | 설명 | 예시 |
|---|---|---|
| 데이터 기반 | 현재 데이터로 답할 수 있어야 함 | 월별 매출은 어떻게 변하는가? |
| 구체적 | 분석 대상과 기준이 명확해야 함 | 카테고리별 매출 비중은 어떻게 다른가? |
| 측정 가능 | 지표로 계산할 수 있어야 함 | 주문 수, 총매출, 평균 구매 금액 |
| 해석 가능 | 결과가 다음 판단과 연결되어야 함 | 어떤 상품군을 더 자세히 봐야 하는가? |
| 검증 가능 | 코드로 확인할 수 있어야 함 | `groupby()`로 계산 가능 |


## 4. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: C:\Users\82102\Documents\Codex\2026-10-10\new-chat\outputs\llm-data-analysis-study
프로젝트 루트: C:\Users\82102\Documents\Codex\2026-10-10\new-chat\outputs\llm-data-analysis-study
전처리 데이터 폴더: C:\Users\82102\Documents\Codex\2026-10-10\new-chat\outputs\llm-data-analysis-study\data\processed
보고서 폴더: C:\Users\82102\Documents\Codex\2026-10-10\new-chat\outputs\llm-data-analysis-study\reports


## 5. 전처리된 데이터 불러오기

6장은 5장에서 저장한 전처리 데이터를 사용합니다. 전처리 파일이 없다면 먼저 아래 명령을 터미널에서 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [2]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('전처리 데이터 불러오기 완료')


전처리 데이터 불러오기 완료


## 6. 데이터 크기와 컬럼 확인하기

EDA를 시작하기 전에 각 데이터의 행/열 수와 컬럼명을 다시 확인합니다. CSV로 저장했다가 다시 불러오면 날짜 컬럼이 문자열로 돌아올 수 있으므로 날짜형 변환도 다시 확인합니다.


In [3]:
data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, df in data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (96, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 7)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

[order_items]
shape: (752, 6)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']



In [4]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

if 'signup_date' in customers.columns:
    customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

if 'order_month' not in orders.columns:
    orders['order_month'] = orders['order_date'].dt.to_period('M').astype(str)

print('order_date 변환 실패:', orders['order_date'].isna().sum())
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())


order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


## 7. 분석 질문을 먼저 표로 정리하기

EDA를 시작하기 전에 어떤 질문을 볼지 먼저 표로 정리하면 분석이 산만해지지 않습니다. 아래 표는 고객, 상품, 주문, 매출, 시간, 고객 가치 관점에서 만들 수 있는 기본 질문입니다.


In [5]:
questions = pd.DataFrame({
    'analysis_area': ['도시·고객', '상품 카테고리', '월별 매출'],
    'question': [
        '도시별 고객 수와 completed 구매 금액은 어떻게 다른가?',
        '상품 수가 많은 카테고리가 completed 매출도 높은가?',
        '월별 completed 매출과 주문당 평균 금액은 어떻게 변하는가?',
    ],
    'analysis_scope': [
        '중복 제거 고객, completed 주문 상세',
        '상품과 completed 주문 상세; 미매칭 상품 별도 범주',
        'completed 주문 상세; 날짜 결측 별도 범주',
    ],
    'metric': [
        '도시별 고객 수, completed line_total 합계',
        '카테고리별 상품 수·판매 수량·completed line_total 합계/비중',
        '월별 completed line_total 합계, 고유 주문 수, 평균 주문 금액',
    ],
    'validation': [
        '도시별 금액과 completed 원본 합계 비교',
        '카테고리별 금액과 completed 원본 합계 비교',
        '월별 금액과 completed 원본 합계 비교',
    ],
})

questions

,analysis_area,question,analysis_scope,metric,validation
0,도시·고객,도시별 고객 수와 completed 구매 금액은 어떻게 다른가?,"중복 제거 고객, completed 주문 상세","도시별 고객 수, completed line_total 합계",도시별 금액과 completed 원본 합계 비교
1,상품 카테고리,상품 수가 많은 카테고리가 completed 매출도 높은가?,상품과 completed 주문 상세; 미매칭 상품 별도 범주,카테고리별 상품 수·판매 수량·completed line_total 합계/비중,카테고리별 금액과 completed 원본 합계 비교
2,월별 매출,월별 completed 매출과 주문당 평균 금액은 어떻게 변하는가?,completed 주문 상세; 날짜 결측 별도 범주,"월별 completed line_total 합계, 고유 주문 수, 평균 주문 금액",월별 금액과 completed 원본 합계 비교


## 8. 고객 데이터 EDA

먼저 고객 데이터만 따로 살펴봅니다. 하나의 데이터셋 안에서 분포와 빈도를 확인하면 이후 병합 분석을 할 때 기준이 생깁니다.


In [6]:
customers['age'].describe()


,age
count,150.000000
mean,43.913333
std,17.802397
min,18.000000
25%,30.000000
50%,41.000000
75%,56.750000
max,150.000000


In [7]:
customer_city = customers['city'].value_counts(dropna=False).reset_index()
customer_city.columns = ['city', 'customer_count']
customer_city['customer_ratio'] = (
    customer_city['customer_count'] / customer_city['customer_count'].sum() * 100
).round(2)

customer_city


,city,customer_count,customer_ratio
0,부산,23,15.33
1,대구,21,14.00
2,광주,21,14.00
3,서울,18,12.00
4,울산,17,11.33
5,인천,16,10.67
6,수원,10,6.67
7,성남,9,6.00
8,고양,8,5.33
9,대전,5,3.33


In [8]:
customer_gender = customers['gender'].value_counts(dropna=False).reset_index()
customer_gender.columns = ['gender', 'customer_count']
customer_gender['customer_ratio'] = (
    customer_gender['customer_count'] / customer_gender['customer_count'].sum() * 100
).round(2)

customer_gender


,gender,customer_count,customer_ratio
0,M,80,53.33
1,F,70,46.67


고객 데이터에서 이어질 수 있는 질문은 다음과 같습니다.

- 고객은 어느 도시에 많이 분포하는가?
- 고객의 평균 나이는 어느 정도인가?
- 성별 고객 수는 어떻게 분포하는가?
- 도시별 구매 금액도 차이가 있는가?


## 9. 상품 데이터 EDA

상품 데이터에서는 카테고리와 가격을 먼저 봅니다. 상품 수가 많은 카테고리가 매출도 높은지는 아직 알 수 없습니다. 그것은 주문 상세와 연결한 뒤 확인해야 합니다.


In [9]:
product_category = products['category'].value_counts(dropna=False).reset_index()
product_category.columns = ['category', 'product_count']
product_category['product_ratio'] = (
    product_category['product_count'] / product_category['product_count'].sum() * 100
).round(2)

product_category


,category,product_count,product_ratio
0,패션,16,16.67
1,뷰티,16,16.67
2,식품,16,16.67
3,도서,13,13.54
4,전자기기,13,13.54
5,스포츠,10,10.42
6,생활용품,9,9.38
7,전자 기기,1,1.04
8,생활 용품,1,1.04
9,뷰 티,1,1.04


In [10]:
products['price'].describe()


,price
count,9.600000e+01
mean,2.048958e+05
std,1.011848e+06
min,5.000000e+03
25%,5.275000e+04
50%,9.450000e+04
75%,1.532500e+05
max,9.999999e+06


In [11]:
category_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        product_count=('product_id', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .assign(avg_price=lambda df: df['avg_price'].round(0))
    .sort_values('avg_price', ascending=False)
)

category_price


,category,product_count,avg_price,min_price,max_price
2,뷰티,16,710937.0,18000.0,9999999.0
3,생활 용품,1,124000.0,124000.0,124000.0
6,식품,16,120812.0,11000.0,195000.0
8,전자기기,13,120231.0,38000.0,178000.0
9,패션,16,115125.0,28000.0,197000.0
0,도서,13,103692.0,34000.0,182000.0
5,스포츠,10,78400.0,5000.0,183000.0
4,생활용품,9,69556.0,29000.0,147000.0
1,뷰 티,1,54000.0,54000.0,54000.0
7,전자 기기,1,21000.0,21000.0,21000.0


## 10. 주문 데이터 EDA

주문 데이터에서는 주문 상태, 결제수단, 주문 기간을 확인합니다. 주문 상태 분포는 이후 매출 분석에서 완료 주문만 볼지, 전체 주문을 볼지 결정하는 기준이 됩니다.


In [12]:
order_status = orders['order_status'].value_counts(dropna=False).reset_index()
order_status.columns = ['order_status', 'order_count']
order_status['order_ratio'] = (
    order_status['order_count'] / order_status['order_count'].sum() * 100
).round(2)

order_status


,order_status,order_count,order_ratio
0,completed,180,60.0
1,cancelled,63,21.0
2,refunded,57,19.0


In [13]:
payment_method = orders['payment_method'].value_counts(dropna=False).reset_index()
payment_method.columns = ['payment_method', 'order_count']
payment_method['order_ratio'] = (
    payment_method['order_count'] / payment_method['order_count'].sum() * 100
).round(2)

payment_method


,payment_method,order_count,order_ratio
0,kakao_pay,90,30.00
1,card,72,24.00
2,bank_transfer,70,23.33
3,naver_pay,68,22.67


In [14]:
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())
print('주문 월 개수:', orders['order_month'].nunique())


주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00
주문 월 개수: 16


## 11. 매출 분석을 위한 데이터 연결

카테고리별 매출을 계산하려면 주문 상세와 상품 정보를 `product_id` 기준으로 연결해야 합니다. 병합 후에는 행 수와 누락 여부를 반드시 확인합니다.


In [15]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_items[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [16]:
completed_orders = orders.loc[orders['order_status'].eq('completed')].copy()
completed_order_items = order_items.loc[
    order_items['order_id'].isin(completed_orders['order_id'])
].copy()

sales_items = completed_order_items.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
)
missing_product_name_count = int(sales_items['product_name'].isna().sum())
missing_category_count = int(sales_items['category'].isna().sum())
print('금액 분석 범위: completed 주문')
print('병합 전 completed order_items:', completed_order_items.shape)
print('병합 후 sales_items:', sales_items.shape)
print('상품명 누락:', missing_product_name_count)
print('카테고리 누락:', missing_category_count)
sales_items['category'] = sales_items['category'].fillna('미매칭 상품')
sales_items.head()

금액 분석 범위: completed 주문
병합 전 completed order_items: (446, 6)
병합 후 sales_items: (446, 9)
상품명 누락: 16
카테고리 누락: 16


,order_item_id,order_id,product_id,quantity,unit_price,line_total,product_name,category,price
0,1,1,39,2.0,172000.0,344000.0,뷰티 상품 039,뷰티,172000.0
1,2,2,26,4.0,79000.0,316000.0,뷰티 상품 026,뷰티,79000.0
2,3,2,46,3.0,87000.0,261000.0,생활용품 상품 046,생활용품,87000.0
3,4,2,80,1.0,187000.0,187000.0,패션 상품 080,패션,187000.0
4,5,2,91,5.0,125000.0,625000.0,전자기기 상품 091,전자기기,125000.0


## 12. 카테고리별 매출 EDA

카테고리별 매출은 어떤 상품군이 매출에 많이 기여했는지 보여 줍니다. 다만 매출이 높은 이유가 판매 수량 때문인지, 단가 때문인지는 추가로 확인해야 합니다.


In [17]:
category_sales = (
    sales_items
    .groupby('category', as_index=False, dropna=False)
    .agg(total_quantity=('quantity', 'sum'), total_sales=('line_total', 'sum'))
    .sort_values('total_sales', ascending=False)
)
category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)
category_sales

,category,total_quantity,total_sales,sales_ratio
10,패션,290.0,32940000.0,23.31
7,식품,226.0,25035000.0,17.71
9,전자기기,176.0,21485000.0,15.20
0,도서,174.0,17663000.0,12.50
3,뷰티,168.0,17103000.0,12.10
6,스포츠,165.0,13237000.0,9.37
5,생활용품,108.0,7001000.0,4.95
1,미매칭 상품,34.0,4015000.0,2.84
4,생활 용품,17.0,2108000.0,1.49
2,뷰 티,12.0,648000.0,0.46


In [18]:
category_sales_with_price = category_sales.merge(
    category_price[['category', 'avg_price']],
    on='category',
    how='left',
)

category_sales_with_price


,category,total_quantity,total_sales,sales_ratio,avg_price
0,패션,290.0,32940000.0,23.31,115125.0
1,식품,226.0,25035000.0,17.71,120812.0
2,전자기기,176.0,21485000.0,15.20,120231.0
3,도서,174.0,17663000.0,12.50,103692.0
4,뷰티,168.0,17103000.0,12.10,710937.0
5,스포츠,165.0,13237000.0,9.37,78400.0
6,생활용품,108.0,7001000.0,4.95,69556.0
7,미매칭 상품,34.0,4015000.0,2.84,NaN
8,생활 용품,17.0,2108000.0,1.49,124000.0
9,뷰 티,12.0,648000.0,0.46,54000.0


해석 예시:

> 관찰: 매출 비중이 높은 카테고리가 존재합니다.
> 가설: 해당 카테고리는 판매 수량이 많거나 평균 단가가 높아서 매출이 클 수 있습니다.
> 추가 분석: 카테고리별 판매 수량과 평균 단가를 함께 비교해야 합니다.


## 13. 월별 매출과 주문 수 EDA

월별 매출을 보려면 주문 상세와 주문 정보를 `order_id` 기준으로 연결합니다. 월별 매출은 시간 흐름을 보여 주지만, 원인을 바로 설명하지는 않습니다.


In [19]:
order_sales = order_items.merge(
    orders,
    on='order_id',
    how='left',
    validate='many_to_one',
)
order_sales = order_sales.loc[order_sales['order_status'].eq('completed')].copy()
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str)
order_sales['order_month'] = order_sales['order_month'].fillna('날짜 확인 필요')
print('금액 분석 범위: completed 주문')
print('completed 주문 상세 행:', len(order_sales))
print('완료 주문의 날짜 누락:', int(order_sales['order_date'].isna().sum()))
print('병합 후 order_sales:', order_sales.shape)
order_sales.head()

금액 분석 범위: completed 주문
completed 주문 상세 행: 446
완료 주문의 날짜 누락: 6
병합 후 order_sales: (446, 12)


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek
0,1,1,39,2.0,172000.0,344000.0,144.0,2026-07-12,naver_pay,completed,2026-07,Sunday
1,2,2,26,4.0,79000.0,316000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday
2,3,2,46,3.0,87000.0,261000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday
3,4,2,80,1.0,187000.0,187000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday
4,5,2,91,5.0,125000.0,625000.0,149.0,2025-10-02,kakao_pay,completed,2025-10,Thursday


In [20]:
monthly_sales = (
    order_sales.groupby('order_month', as_index=False, dropna=False)
    .agg(total_sales=('line_total', 'sum'), order_count=('order_id', 'nunique'))
    .sort_values('order_month')
)
monthly_sales['avg_order_value'] = (
    monthly_sales['total_sales'] / monthly_sales['order_count']
).round(0)
monthly_sales

,order_month,total_sales,order_count,avg_order_value
0,2025-07,9561000.0,11,869182.0
1,2025-08,9362000.0,17,550706.0
2,2025-09,5299000.0,4,1324750.0
3,2025-10,12629000.0,17,742882.0
4,2025-11,9583000.0,16,598938.0
5,2025-12,8581000.0,12,715083.0
6,2026-01,8135000.0,9,903889.0
7,2026-02,7689000.0,12,640750.0
8,2026-03,7247000.0,13,557462.0
9,2026-04,11459000.0,14,818500.0


월별 매출이 특정 월에 높아졌다면 바로 원인을 단정하지 말고, 주문 수가 늘었는지 평균 주문 금액이 늘었는지, 특정 카테고리 매출이 늘었는지 추가로 확인해야 합니다.


## 14. 고객별 구매 금액 EDA

고객별 구매 금액은 우수 고객 분석의 출발점입니다. 하지만 한 번의 고액 구매 고객과 여러 번 반복 구매한 고객은 다르게 해석해야 합니다. 그래서 `total_sales`, `order_count`, `avg_order_value`를 함께 봅니다.


In [21]:
customer_sales_base = order_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='many_to_one',
)
customer_sales_base['city'] = customer_sales_base['city'].fillna('고객 정보 없음')
customer_sales = (
    customer_sales_base.groupby(['customer_id', 'city'], as_index=False, dropna=False)
    .agg(order_count=('order_id', 'nunique'), total_sales=('line_total', 'sum'))
    .sort_values('total_sales', ascending=False)
)
customer_sales['avg_order_value'] = (
    customer_sales['total_sales'] / customer_sales['order_count']
).round(0)
customer_sales.head(10)

,customer_id,city,order_count,total_sales,avg_order_value
1,3.0,서울,3,6702000.0,2234000.0
7,14.0,대구,4,5204000.0,1301000.0
57,90.0,수원,3,4311000.0,1437000.0
59,93.0,울산,4,4001000.0,1000250.0
47,65.0,광주,6,3977000.0,662833.0
66,104.0,서울,2,3966000.0,1983000.0
11,20.0,성남,4,3808000.0,952000.0
27,38.0,성남,4,3656000.0,914000.0
19,28.0,인천,4,3201000.0,800250.0
13,22.0,부산,3,3109000.0,1036333.0


## 15. EDA 결과를 다음 질문으로 연결하기

EDA의 가치는 표를 만드는 데서 끝나지 않습니다. 각 결과가 다음 질문으로 이어져야 합니다.


In [22]:
eda_result_summary = pd.DataFrame({
    'question': [
        '도시별 고객 수와 completed 구매 금액은 어떻게 다른가?',
        '상품 수가 많은 카테고리가 completed 매출도 높은가?',
        '월별 completed 매출과 주문당 평균 금액은 어떻게 변하는가?',
    ],
    'result_table': ['city_customer_sales', 'category_comparison', 'monthly_sales'],
    'next_question': [
        '도시 간 금액 차이는 고객 수와 고객당 금액 중 무엇과 함께 움직이는가?',
        '카테고리 매출 차이는 판매 수량과 상품 가격 중 무엇과 함께 움직이는가?',
        '월별 변화는 주문 수·평균 주문 금액·카테고리 구성 중 무엇과 함께 나타나는가?',
    ],
})

eda_result_summary

,question,result_table,next_question
0,도시별 고객 수와 completed 구매 금액은 어떻게 다른가?,city_customer_sales,도시 간 금액 차이는 고객 수와 고객당 금액 중 무엇과 함께 움직이는가?
1,상품 수가 많은 카테고리가 completed 매출도 높은가?,category_comparison,카테고리 매출 차이는 판매 수량과 상품 가격 중 무엇과 함께 움직이는가?
2,월별 completed 매출과 주문당 평균 금액은 어떻게 변하는가?,monthly_sales,월별 변화는 주문 수·평균 주문 금액·카테고리 구성 중 무엇과 함께 나타나는가?


## 16. EDA 결과 저장하기

EDA 결과표를 CSV로 저장하면 다음 장의 시각화나 보고서 작성에서 다시 사용할 수 있습니다.


In [23]:
customer_city.to_csv(REPORT_DIR / 'ch06_customer_city.csv', index=False, encoding='utf-8-sig')
product_category.to_csv(REPORT_DIR / 'ch06_product_category.csv', index=False, encoding='utf-8-sig')
category_sales.to_csv(REPORT_DIR / 'ch06_category_sales.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv(REPORT_DIR / 'ch06_monthly_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.to_csv(REPORT_DIR / 'ch06_customer_sales.csv', index=False, encoding='utf-8-sig')
eda_result_summary.to_csv(REPORT_DIR / 'ch06_eda_questions.csv', index=False, encoding='utf-8-sig')

list(REPORT_DIR.glob('ch06_*.csv'))


[WindowsPath('C:/Users/82102/Documents/Codex/2026-10-10/new-chat/outputs/llm-data-analysis-study/reports/ch06_category_sales.csv'), WindowsPath('C:/Users/82102/Documents/Codex/2026-10-10/new-chat/outputs/llm-data-analysis-study/reports/ch06_customer_city.csv'), WindowsPath('C:/Users/82102/Documents/Codex/2026-10-10/new-chat/outputs/llm-data-analysis-study/reports/ch06_customer_sales.csv'), WindowsPath('C:/Users/82102/Documents/Codex/2026-10-10/new-chat/outputs/llm-data-analysis-study/reports/ch06_eda_questions.csv'), WindowsPath('C:/Users/82102/Documents/Codex/2026-10-10/new-chat/outputs/llm-data-analysis-study/reports/ch06_monthly_sales.csv'), WindowsPath('C:/Users/82102/Documents/Codex/2026-10-10/new-chat/outputs/llm-data-analysis-study/reports/ch06_product_category.csv')]

## 17. EDA 요약 보고서 만들기

간단한 Markdown 요약 보고서를 만들어 봅니다. 보고서에는 분석 목적, 주요 질문, 핵심 결과, 추가 질문, 해석 시 주의사항을 담습니다.


In [24]:
summary_text = f'''# Chapter 6 EDA 요약 보고서

## 1. 분석 목적

전처리된 온라인 쇼핑몰 데이터를 사용해 고객, 상품, 주문, 매출 관점의 기본 현황을 탐색했습니다.

## 2. 주요 분석 질문

```text
{questions.to_string(index=False)}
```

## 3. 카테고리별 매출 요약

```text
{category_sales.head(10).to_string(index=False)}
```

## 4. 월별 매출 요약

```text
{monthly_sales.to_string(index=False)}
```

## 5. 고객별 구매 금액 상위 10명

```text
{customer_sales.head(10).to_string(index=False)}
```

## 6. 추가 분석 질문

```text
{eda_result_summary.to_string(index=False)}
```

## 7. 해석 시 주의사항

- EDA 결과는 최종 결론이 아니라 추가 분석을 위한 관찰 결과입니다.
- 매출이 높은 카테고리가 반드시 선호도가 높은 카테고리라는 뜻은 아닙니다.
- 월별 매출 변화의 원인을 설명하려면 프로모션, 계절성, 신규 상품 등의 추가 정보가 필요합니다.
- 고객별 구매 금액은 주문 횟수와 평균 주문 금액을 함께 해석해야 합니다.
'''

report_path = REPORT_DIR / 'ch06_eda_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('EDA 요약 보고서 저장 완료:', report_path)


EDA 요약 보고서 저장 완료: C:\Users\82102\Documents\Codex\2026-10-10\new-chat\outputs\llm-data-analysis-study\reports\ch06_eda_summary.md


## 18. 소스 모듈로 같은 EDA 실행하기

위에서 노트북으로 한 단계씩 실행한 EDA는 `src/eda.py`에 함수로 정리되어 있습니다. 반복 실행하거나 프로젝트 코드로 관리하려면 노트북보다 소스 모듈을 사용하는 것이 좋습니다.


In [25]:
from src.eda import (
    build_eda_report,
    load_processed_sales_data,
    run_basic_eda,
    save_eda_outputs,
)

module_data = load_processed_sales_data(PROCESSED_DIR)
module_results = run_basic_eda(module_data)

module_results.keys()


dict_keys(['questions', 'customer_city', 'customer_gender', 'product_category', 'category_price', 'order_status', 'payment_method', 'category_sales', 'monthly_sales', 'customer_sales', 'eda_result_summary'])

In [26]:
module_results['category_sales'].head()


,category,total_quantity,total_sales,sales_ratio
10,패션,290.0,32940000.0,23.31
7,식품,226.0,25035000.0,17.71
9,전자기기,176.0,21485000.0,15.20
0,도서,174.0,17663000.0,12.50
3,뷰티,168.0,17103000.0,12.10


## 19. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 EDA 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/run_eda.py
```

이 스크립트는 6장 EDA 결과 CSV와 `reports/ch06_eda_summary.md`를 자동으로 저장합니다.


## 20. LLM과 함께 질문을 확장하기

LLM은 EDA 질문을 확장하고 결과 해석 문장을 다듬는 데 도움이 됩니다. 하지만 LLM이 제안한 질문이 실제 데이터로 답할 수 있는지는 반드시 사람이 확인해야 합니다.

```text
온라인 쇼핑몰 데이터로 EDA를 수행하려고 합니다.

데이터셋:
- customers: customer_id, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total

요청:
1. 현재 데이터로 검증 가능한 EDA 질문 10개를 만들어 주세요.
2. 각 질문에 필요한 데이터셋과 pandas 기능을 함께 정리해 주세요.
3. 데이터에 없는 내용은 추측하지 마세요.
4. 각 질문을 계산 가능한 지표로 바꿔 주세요.
```


## 21. LLM 결과 해석 요청 예시

EDA 결과 해석을 요청할 때는 원인 단정을 막는 조건을 넣는 것이 좋습니다.

```text
다음은 온라인 쇼핑몰 카테고리별 매출 요약 결과입니다.

category,total_quantity,total_sales,sales_ratio
전자기기,320,12500000,42.5
생활용품,510,7800000,26.5
패션,260,6200000,21.1
식품,430,2900000,9.9

이 결과를 보고서에 넣을 수 있도록 해석해 주세요.

조건:
- 데이터에 없는 원인을 단정하지 말 것
- 원인 설명은 가설로 표현할 것
- 추가로 확인해야 할 분석 질문을 제안할 것
- 관찰, 가설, 추가 분석을 구분할 것
```


## 22. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
3. 결제수단별 평균 주문 금액을 계산하세요.
4. 월별·카테고리별 매출 요약표를 만들어 보세요.
5. 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.
6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.


In [27]:
# 과제 1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.


In [28]:
# 과제 2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.


In [29]:
# 과제 3. 결제수단별 평균 주문 금액을 계산하세요.


In [30]:
# 과제 4. 월별·카테고리별 매출 요약표를 만들어 보세요.


## 실습 과제 결과와 검증

금액 관련 집계는 과제 지침에 따라 모두 `order_status == 'completed'` 범위입니다. 미매칭 상품과 날짜 변환 실패 주문은 별도 범주로 보존해 합계에서 빠지지 않게 합니다.

In [31]:
# 같은 completed 주문 범위의 원본·카테고리·월별 합계를 비교합니다.
completed_source_total = float(order_sales['line_total'].sum())
category_total = float(category_sales['total_sales'].sum())
monthly_total = float(monthly_sales['total_sales'].sum())
total_check = pd.DataFrame({
    'metric': ['completed 원본 상세', '카테고리별 합계', '월별 합계'],
    'total': [completed_source_total, category_total, monthly_total],
})
total_check['difference_from_source'] = total_check['total'] - completed_source_total
total_check['matches'] = np.isclose(total_check['total'], completed_source_total)
total_check

,metric,total,difference_from_source,matches
0,completed 원본 상세,141340000.0,0.0,True
1,카테고리별 합계,141340000.0,0.0,True
2,월별 합계,141340000.0,0.0,True


### 총합 검증 결과

`total_check`에서 세 합계가 일치하는지 확인합니다. 같은 completed 주문 범위로 묶고 미매칭 상품과 날짜 결측을 별도 그룹으로 보존합니다.

### 과제 1. 도시별 고객 수와 completed 구매 금액 비교

In [32]:
# 주문에 고객 도시를 연결합니다. 연결되지 않은 고객은 별도 행으로 둡니다.
city_customer_counts = (
    customers.groupby('city', dropna=False)
    .agg(customer_count=('customer_id', 'nunique')).reset_index()
)
city_customer_counts['city'] = city_customer_counts['city'].fillna('도시 정보 없음')
city_spend_base = order_sales.merge(
    customers[['customer_id', 'city']], on='customer_id', how='left', validate='many_to_one'
)
city_spend_base['city'] = city_spend_base['city'].fillna('고객 정보 없음')
city_completed_sales = (
    city_spend_base.groupby('city', as_index=False, dropna=False)
    .agg(completed_sales=('line_total', 'sum'), purchasing_customers=('customer_id', 'nunique'))
)
city_customer_sales = city_customer_counts.merge(city_completed_sales, on='city', how='outer')
city_customer_sales[['customer_count', 'purchasing_customers', 'completed_sales']] = (
    city_customer_sales[['customer_count', 'purchasing_customers', 'completed_sales']].fillna(0)
)
city_customer_sales = city_customer_sales.sort_values('completed_sales', ascending=False)
city_customer_sales

,city,customer_count,completed_sales,purchasing_customers
6,부산,23.0,24916000.0,16
7,서울,18.0,22211000.0,11
3,광주,21.0,19866000.0,16
11,인천,16.0,15723000.0,12
4,대구,21.0,14781000.0,12
9,수원,10.0,12220000.0,8
10,울산,17.0,11545000.0,9
8,성남,9.0,11288000.0,5
2,고양,8.0,7222000.0,6
0,Unknown,2.0,820000.0,1


### 과제 2. 상품 수와 카테고리 매출 비교

In [33]:
# 카탈로그 내 상품 수와 completed 판매 금액을 비교합니다.
category_product_count = (
    products.groupby('category', as_index=False)
    .agg(product_count=('product_id', 'nunique'))
)
category_comparison = category_product_count.merge(
    category_sales[['category', 'total_quantity', 'total_sales', 'sales_ratio']],
    on='category', how='outer'
)
category_comparison['product_count'] = category_comparison['product_count'].fillna(0).astype(int)
for col in ['total_quantity', 'total_sales', 'sales_ratio']:
    category_comparison[col] = category_comparison[col].fillna(0)
category_comparison = category_comparison.sort_values('total_sales', ascending=False)
category_comparison

,category,product_count,total_quantity,total_sales,sales_ratio
10,패션,16,290.0,32940000.0,23.31
7,식품,16,226.0,25035000.0,17.71
9,전자기기,13,176.0,21485000.0,15.20
0,도서,13,174.0,17663000.0,12.50
3,뷰티,16,168.0,17103000.0,12.10
6,스포츠,10,165.0,13237000.0,9.37
5,생활용품,9,108.0,7001000.0,4.95
1,미매칭 상품,0,34.0,4015000.0,2.84
4,생활 용품,1,17.0,2108000.0,1.49
2,뷰 티,1,12.0,648000.0,0.46


### 과제 3. 결제수단별 평균 주문 금액

In [34]:
# 주문별 금액을 먼저 합산한 뒤 결제수단별 평균·중앙값을 계산합니다.
payment_order_amounts = (
    order_sales.groupby(['payment_method', 'order_id'], as_index=False)
    .agg(order_amount=('line_total', 'sum'))
)
payment_method_aov = (
    payment_order_amounts.groupby('payment_method', as_index=False)
    .agg(order_count=('order_id', 'nunique'),
         avg_order_amount=('order_amount', 'mean'),
         median_order_amount=('order_amount', 'median'))
    .sort_values('avg_order_amount', ascending=False)
)
payment_method_aov[['avg_order_amount', 'median_order_amount']] = (
    payment_method_aov[['avg_order_amount', 'median_order_amount']].round(0)
)
payment_method_aov

,payment_method,order_count,avg_order_amount,median_order_amount
1,card,40,938550.0,693000.0
0,bank_transfer,49,805327.0,777000.0
2,kakao_pay,50,737220.0,635000.0
3,naver_pay,40,686900.0,523000.0


### 과제 4. 월별·카테고리별 completed 매출

In [35]:
# 주문 월과 상품 카테고리를 연결합니다. 미매칭 상품도 별도 범주입니다.
order_sales_category = order_sales.merge(
    products[['product_id', 'category']], on='product_id', how='left', validate='many_to_one'
)
order_sales_category['category'] = order_sales_category['category'].fillna('미매칭 상품')
monthly_category_sales = (
    order_sales_category.groupby(['order_month', 'category'], as_index=False, dropna=False)
    .agg(completed_sales=('line_total', 'sum'),
         total_quantity=('quantity', 'sum'), order_count=('order_id', 'nunique'))
    .sort_values(['order_month', 'completed_sales'], ascending=[True, False])
)
monthly_category_sales

,order_month,category,completed_sales,total_quantity,order_count
7,2025-07,패션,3163000.0,23.0,7
5,2025-07,식품,2123000.0,23.0,5
6,2025-07,전자기기,1319000.0,9.0,3
2,2025-07,뷰티,1133000.0,12.0,5
0,2025-07,도서,1040000.0,11.0,3
3,2025-07,생활용품,406000.0,6.0,2
4,2025-07,스포츠,295000.0,8.0,1
1,2025-07,미매칭 상품,82000.0,1.0,1
14,2025-08,전자기기,2547000.0,20.0,6
15,2025-08,패션,2246000.0,26.0,9


### 과제 5. 반복 구매와 고액 단발 구매를 구분할 기준

In [36]:
# 구매 횟수와 평균 주문 금액을 함께 확인합니다.
customer_order_summary = customer_sales[['customer_id', 'order_count', 'total_sales', 'avg_order_value']].copy()
single_purchase = customer_order_summary.loc[customer_order_summary['order_count'].eq(1)]
single_purchase_cutoff = (
    float(single_purchase['avg_order_value'].quantile(0.90)) if len(single_purchase) else float('inf')
)
customer_order_summary['exploratory_segment'] = np.select(
    [
        customer_order_summary['order_count'].ge(2),
        customer_order_summary['order_count'].eq(1)
        & customer_order_summary['avg_order_value'].ge(single_purchase_cutoff),
    ],
    ['반복 구매', '단발·상위 10% 금액'],
    default='단발·그 외',
)
customer_segment_summary = (
    customer_order_summary.groupby('exploratory_segment', as_index=False)
    .agg(customer_count=('customer_id', 'nunique'),
         median_order_count=('order_count', 'median'),
         median_avg_order_value=('avg_order_value', 'median'),
         total_completed_sales=('total_sales', 'sum'))
    .sort_values('total_completed_sales', ascending=False)
)
print('단발 구매 고객 상위 10% 평균 금액 기준:', round(single_purchase_cutoff, 0))
customer_segment_summary

단발 구매 고객 상위 10% 평균 금액 기준: 1378500.0


,exploratory_segment,customer_count,median_order_count,median_avg_order_value,total_completed_sales
2,반복 구매,53,2.0,777500.0,108740000.0
0,단발·그 외,41,1.0,601000.0,24617000.0
1,단발·상위 10% 금액,5,1.0,1416000.0,7983000.0


### 과제 6. 답할 수 없는 질문을 자료 범위에 맞게 바꾸기

In [37]:
llm_question_prompt = """온라인 쇼핑몰 데이터의 EDA 질문을 확장해 주세요.
원래 질문: 쿠폰 캠페인이 고객의 반복 구매를 늘렸는가?
현재 데이터 컬럼:
- customers: customer_id, gender, age, city, signup_date
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_item_id, order_id, product_id, quantity, unit_price, line_total
- products: product_id, product_name, category, price

요청:
1. 캠페인 노출·쿠폰 사용 필드가 있는지 확인해 답변 가능 여부를 판단해 주세요.
2. 직접 답할 수 없다면 현재 데이터로 답할 수 있는 대체 질문을 제안해 주세요.
3. 필요한 계산과 검증을 제시하고 원인을 단정하지 마세요.
개인 이름이나 행 단위 자료는 사용하지 마세요."""
print(llm_question_prompt)
available_for_cohort = {
    'customers': {'customer_id', 'signup_date'}.issubset(customers.columns),
    'orders': {'order_id', 'customer_id', 'order_date', 'order_status'}.issubset(orders.columns),
    'order_items': {'order_id', 'line_total'}.issubset(order_items.columns),
}
campaign_related_columns = {'campaign_id', 'coupon_id', 'coupon_code', 'coupon_used', 'promotion_id', 'campaign_exposed'}
available_columns = set(customers.columns) | set(products.columns) | set(orders.columns) | set(order_items.columns)
campaign_fields_present = sorted(available_columns & campaign_related_columns)
print('캠페인/쿠폰 관련 컬럼:', campaign_fields_present)
print('대체 코호트 질문에 필요한 컬럼 존재:', available_for_cohort)

온라인 쇼핑몰 데이터의 EDA 질문을 확장해 주세요.
원래 질문: 쿠폰 캠페인이 고객의 반복 구매를 늘렸는가?
현재 데이터 컬럼:
- customers: customer_id, gender, age, city, signup_date
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_item_id, order_id, product_id, quantity, unit_price, line_total
- products: product_id, product_name, category, price

요청:
1. 캠페인 노출·쿠폰 사용 필드가 있는지 확인해 답변 가능 여부를 판단해 주세요.
2. 직접 답할 수 없다면 현재 데이터로 답할 수 있는 대체 질문을 제안해 주세요.
3. 필요한 계산과 검증을 제시하고 원인을 단정하지 마세요.
개인 이름이나 행 단위 자료는 사용하지 마세요.
캠페인/쿠폰 관련 컬럼: []
대체 코호트 질문에 필요한 컬럼 존재: {'customers': True, 'orders': True, 'order_items': True}


## Chapter 06 답안 템플릿

- 이름: 서지민
- GitHub ID: jasminmuze
- 작성일: 2026-10-21
- 최종 Notebook URL: https://github.com/jasminmuze/llm-data-analysis-study/blob/main/chapter06/chapter06.ipynb

### 1. 내가 정한 EDA 질문

위의 `questions` 표에 질문 3개, 범위, 지표, 검증 방법을 정리했습니다. 매출 계산은 `completed` 주문으로 통일했습니다.

### 왜 이 질문을 선택했는가?

도시·카테고리·월은 고객 구성, 상품 운영, 시간 흐름을 살펴보는 기본 축입니다. 현재 고객·주문·상품의 키와 날짜·수량·단가로 계산할 수 있고, 실제 병합과 합계 비교로 검증할 수 있어 선택했습니다.

### 2. 핵심 집계 결과

- completed 원본 합계: `141,340,000`원
- 카테고리별 합계: `141,340,000`원
- 월별 합계: `141,340,000`원
- 총합 일치 여부: `일치 (차이 0원)`

![completed 카테고리·월별 EDA 결과](images/step02_eda.svg)

### 결과 관찰

- 카테고리 집계에서 패션이 32,940,000원(23.31%)으로 가장 컸고, 식품 25,035,000원(17.71%), 전자기기 21,485,000원(15.20%) 순이었습니다. 패션·식품·뷰티는 카탈로그 상품 수가 각각 16개였지만 판매 금액은 달랐습니다. 상품 정보가 연결되지 않은 상세 16건, 4,015,000원(2.84%)은 ‘미매칭 상품’으로 남았습니다. 카테고리 표기가 ‘뷰티/뷰 티’, ‘전자기기/전자 기기’, ‘생활용품/생활 용품’처럼 나뉘어 있어 표준화 기준도 필요합니다.
- 부산은 고객 23명, completed 구매 금액 24,916,000원으로 도시 중 금액이 가장 컸고, 서울은 고객 18명, 22,211,000원이었습니다. ‘Unknown’ 도시 고객 2명은 820,000원을 구매했고, 고객 테이블과 연결되지 않은 주문 376,000원은 별도 구분했습니다.
- 유효한 날짜가 있는 월 중 2026년 8월 금액이 16,980,000원(주문 15건, 주문당 평균 1,132,000원)으로 가장 컸습니다. 날짜 변환이 안 된 완료 주문 2건의 상세 6행, 1,903,000원은 ‘날짜 확인 필요’로 보존했습니다. 월별 금액만으로 상승 원인을 단정할 수 없습니다.

### 나의 해석과 판단

카테고리별 매출 차이는 판매 수량과 상품 가격을 함께 보아야 설명할 수 있습니다. 월별 금액 변화도 주문 수와 주문당 평균 금액을 분리해 확인해야 합니다. 이 표본만으로 구매 선호나 변화 원인을 단정하지 않았습니다.

### 업무·분석적 의미

상품 구색과 재고 검토는 카탈로그의 상품 수만으로 정하지 않고 completed 판매 수량·금액과 함께 살펴야 합니다. 월별 모니터링은 총액 외에 주문 수와 평균 주문 금액도 나란히 확인하는 편이 적절합니다.

### 한계와 추가 확인 사항

날짜 변환 실패 행은 월별 비교에서 별도 범주로 보존했고 미매칭 상품도 별도로 집계했습니다. 완료 상태만으로 할인·배송비·반품 후 정산이 반영됐다고 볼 수 없어 `line_total`을 순매출로 단정할 수 없습니다. 프로모션·재고·유입 경로 자료도 없어 원인을 말할 수 없습니다.

### 3. 관찰 → 가설 → 추가 검증

#### 사례 1
- 관찰: 패션 카테고리 금액은 32,940,000원(23.31%)으로 가장 컸고, 상품 수가 16개로 같은 식품·뷰티의 금액은 각각 25,035,000원과 17,103,000원이었습니다. 미매칭 상품 16개 상세 행의 금액 4,015,000원도 별도 집계했습니다.
- 가설: 상품 가격대 또는 구매 수량 차이가 카테고리별 금액 차이와 함께 나타났을 수 있습니다.
- 추가 검증: 카테고리별 단가 분포, 주문당 수량, 할인·반품 포함 여부를 비교합니다.

#### 사례 2
- 관찰: 2026년 8월 completed 금액이 16,980,000원으로 유효 날짜 월 중 가장 컸습니다(주문 15건, 주문당 평균 1,132,000원). 날짜를 확인할 수 없는 완료 주문 2건의 1,903,000원은 별도 월 범주로 두었습니다.
- 가설: 월별 금액 차이가 주문 수와 주문당 평균 금액 중 하나 또는 둘 모두와 관련됐을 수 있습니다.
- 추가 검증: 월별 주문 수, 평균 주문 금액, 카테고리 구성을 비교하고 날짜 유효 기간을 확인합니다.

#### 사례 3
- 관찰: 반복 구매 고객은 53명으로 집계되었고 completed 금액 합계는 108,740,000원이었습니다. 단발 구매 고객 중 평균 주문 금액 상위 10% 기준(1,378,500원)에 해당한 고객은 5명, 금액 합계는 7,983,000원이었습니다.
- 가설: 한 번의 큰 주문이 누적 금액 상위 고객을 만들 수 있어 반복 구매와는 다른 유형일 수 있습니다.
- 추가 검증: 주문 횟수와 평균 금액으로 구간을 비교하고 관측 기간·환불·할인 정보를 확인합니다.

### 4. LLM 질문 확장

- Safe Context: 데이터셋 이름과 컬럼 목록만 제공했습니다. 개인 이름이나 행 단위 자료는 Prompt에 넣지 않았습니다.
- Prompt: `llm_question_prompt` 코드 셀의 출력
- LLM 제안 요약: 캠페인 노출·쿠폰 사용 필드가 없어 캠페인 효과는 현재 자료로 식별할 수 없습니다. 대체로 가입 월 코호트별 가입 후 90일 내 completed 매출과 재구매율을 비교하되, 이는 캠페인 효과가 아닌 코호트 간 기술적 비교라고 제안합니다.
- 실제 사용 여부: 수정 후 사용
- 판단 이유: 가입일·고객 ID·주문일·완료 상태·상세 금액 필드는 존재하지만 코호트별 90일 관측 기간을 확인해야 합니다. 캠페인 노출 변수가 없으므로 결과를 마케팅 효과로 해석할 수 없습니다.
- 사람이 수정한 내용: 날짜 변환 실패와 고객 키 연결을 먼저 확인하고, 가입 후 90일 관측이 가능한 고객만 비교하도록 제한합니다. 필요하면 캠페인 배정 자료를 추가 요청합니다.

![LLM 질문 확장 검토](images/step04_llm.svg)

### 5. 다음 분석 우선순위

1. 날짜 오류와 주문·상품·고객 키 연결 문제를 원천 자료에서 확인합니다.
2. 카테고리별 판매 수량과 가격·할인 정보를 나누어 매출 차이를 검증합니다.
3. 가입 코호트별 첫 90일 구매를 비교하고 캠페인 노출 자료가 있는지 확인합니다.

#### 가장 먼저 확인하고 싶은 이유

날짜와 키 연결이 잘못되면 월별 추세, 고객별 구매, 카테고리별 합계가 모두 어긋날 수 있으므로 해석 전에 계산 범위를 점검해야 합니다.

#### 현재 데이터만으로 단정할 수 없는 것

매출 변화 원인, 광고·쿠폰 효과, 고객 선호·이탈 이유, `line_total`이 회계상 순매출과 같은지는 이 CSV만으로 확정할 수 없습니다.

### 최종 체크

- [x] 질문과 지표가 연결되어 있습니다.
- [x] completed 주문 범위로 집계 총합을 검증했습니다.
- [x] 관찰과 가설을 구분했습니다.
- [x] 추가 검증 질문을 작성했습니다.
- [x] LLM 제안을 실제 필드로 검토했습니다.
- [x] 최종 Notebook URL을 제출합니다.

## 23. 정리

이번 장에서는 다음 내용을 실습했습니다.

- EDA에서 관찰, 가설, 결론 구분하기
- 현재 데이터로 답할 수 있는 질문 만들기
- 고객, 상품, 주문 데이터의 기본 분포 확인
- 주문 상세, 상품, 주문, 고객 데이터를 병합해 매출 분석하기
- 카테고리별 매출, 월별 매출, 고객별 구매 금액 계산하기
- EDA 결과를 다음 질문으로 연결하기
- EDA 결과 CSV와 Markdown 보고서 저장하기
- `src/eda.py`와 `scripts/run_eda.py`로 반복 실행 가능한 소스 구조 만들기

다음 장에서는 EDA에서 만든 질문과 집계표를 바탕으로 데이터 시각화를 다룹니다.
